# Import + data + preprocs (before starting enable gpu and after you finish disable it)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers

import warnings, os
warnings.filterwarnings("ignore")

# Reproducibility for the gpu thing in kaggle, an actual W feature to accelerate training and testing of DL models
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

# 1.LOAD DATA============================================================================================================

import glob

parquet_files = glob.glob("/kaggle/input/notebooks/dhoogla/cse-cic-ids2018-00-cleaning/*.parquet")
print(f"Found {len(parquet_files)} files")

df = pd.concat([pd.read_parquet(f) for f in parquet_files], ignore_index=True)
print(f"Dataset shape: {df.shape}")
print(df.dtypes.value_counts())
print("\nLabel distribution:\n", df["Label"].value_counts())

# 2. PREPROCESSING============================================================================================================

# 2a. Drop columns that are pure metadata or all-zeros to not make them as rubbish
cols_to_drop = ["Timestamp"]          # add others you don't want
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

# 2b. Replace inf / -inf with NaN then drop cuz they wont be necessary
df.replace([np.inf, -np.inf], np.nan, inplace=True)
before = len(df)
df.dropna(inplace=True)
print(f"Rows after dropping NaN/Inf: {len(df):,}  (removed {before - len(df):,})")

# 2c. Separate features / labels
LABEL_COL = "Label"
X = df.drop(columns=[LABEL_COL]).select_dtypes(include=[np.number]).values.astype(np.float32)
y_raw = df[LABEL_COL].values

# 2d. Encode labels
le = LabelEncoder()
y = le.fit_transform(y_raw)
NUM_CLASSES = len(le.classes_)
print(f"\nClasses ({NUM_CLASSES}):", le.classes_)

# 2e. (Optional) subsample for speed — remove or increase for full training
# e.g. keep 1M rows
MAX_SAMPLES = 1_000_000
if len(X) > MAX_SAMPLES:
    idx = np.random.choice(len(X), MAX_SAMPLES, replace=False)
    X, y = X[idx], y[idx]
    print(f"Subsampled to {MAX_SAMPLES:,} rows")

# 2f. Train / val / test split  (70 / 15 / 15)
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)

print(f"\nTrain: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

# 2g. Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

NUM_FEATURES = X_train.shape[1]
print(f"Number of features: {NUM_FEATURES}")

# 2h. Class weights (handle imbalance without oversampling)
class_weights_arr = compute_class_weight(
    "balanced", classes=np.arange(NUM_CLASSES), y=y_train)
CLASS_WEIGHTS = dict(enumerate(class_weights_arr))
print("\nClass weights (truncated):", {k: round(v, 2) for k, v in CLASS_WEIGHTS.items()})

## small note:
there are some classes here that have less than 100 so its basiclly dead weight for tese models, as it either increases the acc too much or punishes it too much.
the best approach is to remove them to no affect the trainng process:

these one in pirticular:

SQL Injection                    85

DoS attacks-SlowHTTPTest         55

FTP-BruteForce                   53

## After 1M subsampling, top 5 lowest classes fell below the 100-sample threshold and
 were automatically dropped during preprocessing:
   - SQL Injection        (85  total samples) and way lower after subsampling
   - DoS attacks-SlowHTTPTest (55  total samples) and way lower after subsampling
   - FTP-BruteForce       (53  total samples) and way lower after subsampling
   - Brute Force -Web     (568 total → <100 after subsampling)
   - Brute Force -XSS     (229 total → <100 after subsampling)
 Final dataset: 10 classes used for all models (classical ML + NN)

so even if the naming is Classe_12 its technically 10 because -web and -xss BF got droped after subsampling

In [ ]:
MIN_CLASS_SAMPLES = 100

# Use y (already subsampled to 1M) not y_raw
class_counts = pd.Series(le.inverse_transform(y)).value_counts()
valid_classes = class_counts[class_counts >= MIN_CLASS_SAMPLES].index.tolist()

print(f"Keeping {len(valid_classes)} classes:")
print(sorted(valid_classes))
dropped = [c for c in le.classes_ if c not in valid_classes]
print(f"Dropping: {dropped}")

# Build mask from y (same size as X = 1M)
y_labels = le.inverse_transform(y)
mask = pd.Series(y_labels).isin(valid_classes).values

X_filtered = X[mask]
y_raw_filtered = y_labels[mask]

# Re-encode with new label encoder
le2 = LabelEncoder()
y_filtered = le2.fit_transform(y_raw_filtered)
NUM_CLASSES_2 = len(le2.classes_)
print(f"\nNew label set ({NUM_CLASSES_2} classes):", le2.classes_)

# Re-split
X_tr2, X_tmp2, y_tr2, y_tmp2 = train_test_split(
    X_filtered, y_filtered, test_size=0.30, random_state=SEED, stratify=y_filtered)
X_v2, X_te2, y_v2, y_te2 = train_test_split(
    X_tmp2, y_tmp2, test_size=0.50, random_state=SEED, stratify=y_tmp2)

# Re-scale
scaler2 = StandardScaler()
X_tr2 = scaler2.fit_transform(X_tr2)
X_v2  = scaler2.transform(X_v2)
X_te2 = scaler2.transform(X_te2)

# Subsample for classical ML
idx2 = np.random.choice(len(X_tr2), min(300_000, len(X_tr2)), replace=False)
X_tr_cl2, y_tr_cl2 = X_tr2[idx2], y_tr2[idx2]

print(f"\nTrain: {X_tr2.shape}  Val: {X_v2.shape}  Test: {X_te2.shape}")

# first ML models -> RF / LighGBM aka 'GB' / XGB  (initial + tuned)

In [ ]:
# ── CLASSICAL ML MODELS (FIXED) ───────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import xgboost as xgb
import lightgbm as lgb

CLASSICAL_SAMPLES = 300_000
idx_cl = np.random.choice(len(X_train), CLASSICAL_SAMPLES, replace=False)
X_tr_cl = X_train[idx_cl]
y_tr_cl = y_train[idx_cl]

# ── Random Forest ──────────────────────────────────────────────────────────────
print("Training Random Forest...")
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=4,
    class_weight="balanced",
    n_jobs=-1,
    random_state=SEED
)
rf.fit(X_tr_cl, y_tr_cl)
y_pred_rf = rf.predict(X_test)
print("\n── Random Forest ──")
print(classification_report(y_test, y_pred_rf, target_names=le.classes_, zero_division=0))


print("Training Random Forest (tuned + useless class dropped)...")
rf2 = RandomForestClassifier(
    n_estimators=300,        # more trees
    max_depth=25,            # slightly deeper
    min_samples_leaf=2,      # finer splits
    max_features="sqrt",     # standard for classification
    class_weight="balanced_subsample",  # better than "balanced" for RF
    n_jobs=-1,
    random_state=SEED
)
rf2.fit(X_tr_cl2, y_tr_cl2)
y_pred_rf2 = rf2.predict(X_te2)
print("\n── Random Forest (tuned) ──")
print(classification_report(y_te2, y_pred_rf2, target_names=le2.classes_, zero_division=0))

In [ ]:
# ── XGBoost (fixed — better minority class handling) ──────────────────────────
print("\nTraining XGBoost...")

# Per-class sample weights → replaces scale_pos_weight for multiclass
from sklearn.utils.class_weight import compute_sample_weight
sample_weights_xgb = compute_sample_weight("balanced", y_tr_cl)

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,          # reduced from 8 → less overfitting
    learning_rate=0.05,   # slower learning → better generalization
    subsample=0.7,        # reduced from 0.8
    colsample_bytree=0.7,
    min_child_weight=5,   # prevents splits on very rare samples
    gamma=1.0,            # minimum loss reduction to split — key regularizer
    reg_alpha=0.1,        # L1 regularization
    reg_lambda=2.0,       # L2 regularization
    eval_metric="mlogloss",
    tree_method="hist",
    device="cuda",
    random_state=SEED,
    early_stopping_rounds=20,
)
xgb_model.fit(
    X_tr_cl, y_tr_cl,
    sample_weight=sample_weights_xgb,   # pass class weights here
    eval_set=[(X_val, y_val)],
    verbose=50
)
y_pred_xgb = xgb_model.predict(X_test)
print("\n── XGBoost ──")
print(classification_report(y_test, y_pred_xgb, target_names=le.classes_, zero_division=0))

# ── FIX 3: XGBoost — fix the Benign recall collapse ───────────────────────────
# Instead of full balanced weights, use softer custom weights
from sklearn.utils.class_weight import compute_class_weight

class_w = compute_class_weight("balanced", classes=np.unique(y_tr_cl2), y=y_tr_cl2)

# Soft-cap weights: don't let any class get more than 20x the majority weight
max_weight = class_w.min() * 20
class_w_capped = np.clip(class_w, None, max_weight)

sample_w2 = class_w_capped[y_tr_cl2]

print("Training XGBoost (tuned + useless class dropped)...")
xgb2 = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,    # increased → more conservative splits
    gamma=2.0,              # increased → harder to split
    reg_alpha=0.5,
    reg_lambda=3.0,
    eval_metric="mlogloss",
    tree_method="hist",
    device="cuda",
    random_state=SEED,
    early_stopping_rounds=30,
)
xgb2.fit(
    X_tr_cl2, y_tr_cl2,
    sample_weight=sample_w2,
    eval_set=[(X_v2, y_v2)],
    verbose=50
)
y_pred_xgb2 = xgb2.predict(X_te2)
print("\n── XGBoost (tuned) ──")
print(classification_report(y_te2, y_pred_xgb2, target_names=le2.classes_, zero_division=0))

In [ ]:
print("\nTraining LightGBM...")
lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    num_leaves=31,          # ← reduced from 63, prevents over-splitting
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    min_child_samples=50,   # ← increased from 5, stops splits on tiny groups
    min_split_gain=0.01,    # ← minimum gain required to make a split
    reg_alpha=0.1,          # ← L1 regularization
    reg_lambda=1.0,         # ← L2 regularization
    device="cpu",
    verbose=-1,             # ← silences the warning spam completely
    random_state=SEED,
    n_jobs=-1
)
lgb_model.fit(
    X_tr_cl, y_tr_cl,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)]
)
y_pred_lgb = lgb_model.predict(X_test)
print("\n── LightGBM ──")
print(classification_report(y_test, y_pred_lgb, target_names=le.classes_, zero_division=0))


# ── FIX 4: LightGBM — needs more rounds + better Infilteration handling ────────
print("\nTraining LightGBM (tuned + useless class dropped)...")
lgb2 = lgb.LGBMClassifier(
    n_estimators=1000,       # was 500, didn't converge
    max_depth=10,
    learning_rate=0.03,      # slower → better generalization
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    min_child_samples=20,    # less strict than 50 → catches Infilteration better
    min_split_gain=0.005,    # slightly lower threshold
    reg_alpha=0.1,
    reg_lambda=1.0,
    device="cpu",
    verbose=-1,
    random_state=SEED,
    n_jobs=-1
)
lgb2.fit(
    X_tr_cl2, y_tr_cl2,
    eval_set=[(X_v2, y_v2)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]  # more patience
)
y_pred_lgb2 = lgb2.predict(X_te2)
print("\n── LightGBM (tuned) ──")
print(classification_report(y_te2, y_pred_lgb2, target_names=le2.classes_, zero_division=0))

# comparison + confusion matrixes of all 3 ML models

In [ ]:
# ── ALL 6 CONFUSION MATRICES ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(30, 16))
fig.suptitle("Confusion Matrices — Baseline vs Tuned", fontsize=16,
             fontweight="bold", y=1.01)

# Row 1: baseline (15 classes, original le)
baseline_cms = [
    ("RF (baseline)",       rf.predict(X_test),        y_test, le.classes_),
    ("XGBoost (baseline)",  xgb_model.predict(X_test), y_test, le.classes_),
    ("LightGBM (baseline)", lgb_model.predict(X_test), y_test, le.classes_),
]
# Row 2: tuned (12 classes, le2)
tuned_cms = [
    ("RF (tuned)",       rf2.predict(X_te2),  y_te2, le2.classes_),
    ("XGBoost (tuned)",  xgb2.predict(X_te2), y_te2, le2.classes_),
    ("LightGBM (tuned)", lgb2.predict(X_te2), y_te2, le2.classes_),
]

for col, (name, y_pred, y_true, classes) in enumerate(baseline_cms):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=classes)
    disp.plot(ax=axes[0, col], xticks_rotation=45,
              colorbar=False, cmap="Reds")
    axes[0, col].set_title(name, fontsize=12, fontweight="bold", pad=10)
    axes[0, col].tick_params(axis="both", labelsize=7)

for col, (name, y_pred, y_true, classes) in enumerate(tuned_cms):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=classes)
    disp.plot(ax=axes[1, col], xticks_rotation=45,
              colorbar=False, cmap="Blues")
    axes[1, col].set_title(name, fontsize=12, fontweight="bold", pad=10)
    axes[1, col].tick_params(axis="both", labelsize=7)

# Row labels
axes[0, 0].set_ylabel("Baseline\n\nTrue Label", fontsize=11, fontweight="bold")
axes[1, 0].set_ylabel("Tuned\n\nTrue Label", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("confusion_matrices_all6.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
baseline_results = {}
for name, model, Xte, yte in [
    ("RF (baseline)",      rf,        X_test, y_test),
    ("XGBoost (baseline)", xgb_model, X_test, y_test),
    ("LightGBM (baseline)",lgb_model, X_test, y_test),
]:
    preds = model.predict(Xte)
    baseline_results[name] = round((preds == yte).mean() * 100, 2)

tuned_results = {}
for name, model, Xte, yte in [
    ("RF (tuned)",       rf2,   X_te2, y_te2),
    ("XGBoost (tuned)",  xgb2,  X_te2, y_te2),
    ("LightGBM (tuned)", lgb2,  X_te2, y_te2),
]:
    preds = model.predict(Xte)
    tuned_results[name] = round((preds == yte).mean() * 100, 2)

all_classical = {**baseline_results, **tuned_results}

print("\n" + "="*45)
print("   Classical ML — Baseline vs Tuned")
print("="*45)
for name, acc in all_classical.items():
    bar = "█" * int(acc / 2)
    print(f"  {name:<25}: {acc:.2f}%  {bar}")
print("="*45)

# Grouped bar chart
labels    = ["Random Forest", "XGBoost", "LightGBM"]
baseline  = list(baseline_results.values())
tuned     = list(tuned_results.values())
x         = np.arange(len(labels))
width     = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, baseline, width, label="Baseline",
               color=["#4C72B0", "#DD8452", "#55A868"], alpha=0.6, edgecolor="white")
bars2 = ax.bar(x + width/2, tuned,    width, label="Tuned",
               color=["#4C72B0", "#DD8452", "#55A868"], alpha=1.0, edgecolor="white")

ax.set_ylabel("Test Accuracy (%)", fontsize=12)
ax.set_title("Classical ML — Baseline vs Tuned", fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 108)
ax.legend(fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_axisbelow(True)

for bar, val in zip(bars1, baseline):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.3, f"{val}%", ha="center", fontsize=9)
for bar, val in zip(bars2, tuned):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.3, f"{val}%", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig("classical_comparison_all6.png", dpi=120, bbox_inches="tight")
plt.show()

# 1st NN model -> 1 dim CNN (fine tuned)

## the helpers function for the nn based models + dropped the 15 classes to 10 (explanation above the ML codes), just like the previous finetuned ML models

In [ ]:
class_weights_arr2 = compute_class_weight(
    "balanced", classes=np.arange(NUM_CLASSES_2), y=y_tr2)
CLASS_WEIGHTS2 = dict(enumerate(class_weights_arr2))

def compile_and_train(model, X_tr2, y_tr2, X_v2, y_v2, model_name,
                      epochs=30, batch_size=4096, lr=1e-3, class_weight=None):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    model.summary()
    cb = [
        callbacks.EarlyStopping(patience=10, restore_best_weights=True,
                                monitor="val_loss"),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-6)
    ]
    history = model.fit(
        X_tr2, y_tr2,
        validation_data=(X_v2, y_v2),
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weight if class_weight is not None else CLASS_WEIGHTS2, # this specificly for the CNN since my god this NN is deadass a beta :pray:
        callbacks=cb,
        verbose=1
    )
    return history

def evaluate_model(model, X_te2, y_te2, model_name):
    y_pred = np.argmax(model.predict(X_te2, batch_size=8192), axis=1)
    print(f"\n{'='*60}")
    print(f" {model_name} — Test Results")
    print('='*60)
    print(classification_report(y_te2, y_pred,
                                target_names=le2.classes_,   
                                zero_division=0))
    cm = confusion_matrix(y_te2, y_pred)
    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(cm, display_labels=le2.classes_)  
    disp.plot(ax=ax, xticks_rotation=45, colorbar=False, cmap="Blues")
    ax.set_title(f"{model_name} — Confusion Matrix")
    plt.tight_layout()
    plt.savefig(f"{model_name.replace(' ', '_')}_cm.png", dpi=100)
    plt.show()
    return y_pred

def plot_history(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history["loss"], label="Train")
    axes[0].plot(history.history["val_loss"], label="Val")
    axes[0].set_title(f"{model_name} — Loss")
    axes[0].legend()
    axes[1].plot(history.history["accuracy"], label="Train")
    axes[1].plot(history.history["val_accuracy"], label="Val")
    axes[1].set_title(f"{model_name} — Accuracy")
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(f"{model_name.replace(' ', '_')}_history.png", dpi=100)
    plt.show()

In [ ]:
# 1. 1D-CNN 
import keras_tuner as kt # for tunning

def build_cnn(input_dim, num_classes):
    inp = layers.Input(shape=(input_dim, 1))

    x = layers.Conv1D(64, kernel_size=5, padding="same", activation="relu")(inp)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Conv1D(128, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Conv1D(64, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)

    return models.Model(inp, out, name="CNN_1D")

# Lighter class weights for CNN (BN instability fix also to try to fix the low accuracy)
cw_light = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES_2), y=y_tr2)
cw_light_capped = np.clip(cw_light, None, cw_light.min() * 5)  # 5x instead of 15x
CLASS_WEIGHTS_CNN = dict(enumerate(cw_light_capped))
print("CNN class weights:", {k: round(v, 2) for k, v in CLASS_WEIGHTS_CNN.items()})


# Reshape for CNN: (samples, features, 1)
X_train_cnn = X_tr2[..., np.newaxis]
X_val_cnn   = X_v2[..., np.newaxis]
X_test_cnn  = X_te2[..., np.newaxis]

cnn_model = build_cnn(NUM_FEATURES, NUM_CLASSES_2)
cnn_history = compile_and_train(
    cnn_model, X_train_cnn, y_tr2, X_val_cnn, y_v2, "CNN", epochs=50, lr=3e-4, class_weight=CLASS_WEIGHTS_CNN)

plot_history(cnn_history, "CNN")
evaluate_model(cnn_model, X_test_cnn, y_te2, "CNN")

cnn_model.save("cnn.keras")

# 2nd NN model: RNN-LSTM

In [ ]:
# 2.LSTM (RNN) 
import keras_tuner as kt
CLASS_WEIGHTS_LIGHT = CLASS_WEIGHTS_CNN  # reuse the same 5x capped weights

def build_lstm(input_dim, num_classes):
    inp = layers.Input(shape=(input_dim, 1))

    x = layers.Bidirectional(
        layers.LSTM(64, return_sequences=True))(inp)
    x = layers.Dropout(0.3)(x)

    x = layers.Bidirectional(
        layers.LSTM(32, return_sequences=False))(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(num_classes, activation="softmax")(x)

    return models.Model(inp, out, name="BiLSTM_RNN")


# Reuse CNN-reshaped arrays
lstm_model = build_lstm(NUM_FEATURES, NUM_CLASSES_2)
lstm_history = compile_and_train(
    lstm_model, X_train_cnn, y_tr2, X_val_cnn, y_v2, "RNN_LSTM",  
    epochs=30, batch_size=4096)

plot_history(lstm_history, "RNN_LSTM")
evaluate_model(lstm_model, X_test_cnn, y_te2, "RNN_LSTM")  

lstm_model.save("lstm.keras")

# 3nd NN model: MLP

In [ ]:
# 3. MLP 
CLASS_WEIGHTS_LIGHT = CLASS_WEIGHTS_CNN  # reuse the same 5x capped weights

def build_mlp(input_dim, num_classes):
    inp = layers.Input(shape=(input_dim,))
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=regularizers.l2(1e-4))(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, activation="relu",
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.2)(x)

    out = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model(inp, out, name="MLP")


mlp_model = build_mlp(NUM_FEATURES, NUM_CLASSES_2) 
mlp_history = compile_and_train(
    mlp_model, X_tr2, y_tr2, X_v2, y_v2, "MLP", epochs=50, class_weight=CLASS_WEIGHTS_LIGHT)

plot_history(mlp_history, "MLP")
evaluate_model(mlp_model, X_te2, y_te2, "MLP")

mlp_model.save("mlp.keras")

# 3 way NN Models Comparaison

In [ ]:
# NN based models COMPARISON
results = {}
for name, model, Xte in [("MLP",  mlp_model,  X_te2),
                          ("CNN",  cnn_model,  X_test_cnn),
                          ("LSTM", lstm_model, X_test_cnn)]:
    y_pred = np.argmax(model.predict(Xte, batch_size=8192), axis=1)
    acc = (y_pred == y_te2).mean()          # ← y_test → y_te2
    results[name] = round(acc * 100, 2)

print("\n" + "="*40)
print("  NN Models Accuracy Comparison")
print("="*40)
for name, acc in results.items():
    print(f"  {name:<8}: {acc:.2f}%")
print("="*40)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(results.keys(), results.values(), color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_ylabel("Test Accuracy (%)")
ax.set_title("NN Models — CSE-CIC-IDS2018 (12 classes)")  # ← updated title
ax.set_ylim(0, 105)
for bar, val in zip(ax.patches, results.values()):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5, f"{val}%", ha="center")
plt.tight_layout()
plt.savefig("nn_model_comparison.png", dpi=100)
plt.show()

# ── SAVE MODELS ───────────────────────────────────────────────────────────────
mlp_model.save("mlp.keras") 
cnn_model.save("cnn.keras")
lstm_model.save("lstm.keras")
print("\nModels saved.")


**before we move to sub optimal solutions, here are some quick marks about the results of the models:**

**1-CNN, MLP and rf tuned performed very well [98,98,95], tho the CNN took very significant time loss while traning unlike the RF and MLP**

**2-xgboost shows a very nice accuracy flow with a much much lower time in training compared to both rf and cnn**

**3-Gradiantboost had the weakest wheel of tunes in all showing unsignificant boost after getting fine tuned and dropped unreliable classes to be trained with, barely a 1% boost, so meaningless but still decent reuslts.**

**4- and the RNN or LSTM is basicly a model in a wheel chair, we experimented with it so long yet the best was jsut 67%, very terrible.**

**more reasons on why tho, take a look:**

RNN/LSTM sucks for 3 reasons:

Same class weight problem as the original MLP/CNN Benign recall 0.59, but fixing it would require retraining and it already took ~17 minutes per run


LSTM is fundamentally the wrong architecture for this data — flow statistics are not sequential data. There's no temporal order to the 77 features, so the LSTM's memory mechanism adds complexity without benefit. That's why CNN and MLP both demolished it


It still serves a purpose in your comparison — showing that LSTM underperforms on tabular/network flow data is actually a valid and interesting finding worth keeping

the comparison graph is down below along with the next alternatives called hybrid models.


# Hybrid model 1 : XG x CNN

In [ ]:
# ── HYBRID 1: XGBoost + CNN ───────────────────────────────────────────────────

# Step 1: CNN embeddings — uses the same cnn_model that got 98%
cnn_feat_extractor = models.Model(
    inputs=cnn_model.input,
    outputs=cnn_model.layers[-2].output  # layer before softmax
)
print("Extracting CNN embeddings...")
cnn_train_emb = cnn_feat_extractor.predict(X_train_cnn, batch_size=8192)
cnn_val_emb   = cnn_feat_extractor.predict(X_val_cnn,   batch_size=8192)
cnn_test_emb  = cnn_feat_extractor.predict(X_test_cnn,  batch_size=8192)

# Step 2: retrain XGB on full filtered 10-class data
print("Retraining XGBoost for hybrid...")
xgb_full = xgb.XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="mlogloss", tree_method="hist", device="cuda",
    random_state=SEED, early_stopping_rounds=20
)
xgb_full.fit(X_tr2, y_tr2, eval_set=[(X_v2, y_v2)], verbose=50)

xgb_train_prob = xgb_full.predict_proba(X_tr2)
xgb_val_prob   = xgb_full.predict_proba(X_v2)
xgb_test_prob  = xgb_full.predict_proba(X_te2)

# Step 3: concatenate CNN embeddings + XGB probs
X_hyb1_train = np.concatenate([cnn_train_emb, xgb_train_prob], axis=1)
X_hyb1_val   = np.concatenate([cnn_val_emb,   xgb_val_prob],   axis=1)
X_hyb1_test  = np.concatenate([cnn_test_emb,  xgb_test_prob],  axis=1)

# Step 4: meta-head — no BN (same lesson learned from CNN)
def build_meta_head(input_dim, num_classes):
    inp = layers.Input(shape=(input_dim,))
    x = layers.Dense(128, activation="relu")(inp)
    x = layers.Dropout(0.3)(x)              # ← removed BN, dropout only
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model(inp, out, name="XGB_CNN_Hybrid")

hybrid1 = build_meta_head(X_hyb1_train.shape[1], NUM_CLASSES_2)  # ← NUM_CLASSES_2
hybrid1.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4),  # ← same lr as CNN
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
hybrid1.fit(
    X_hyb1_train, y_tr2,                       # ← y_train → y_tr2
    validation_data=(X_hyb1_val, y_v2),        # ← y_val → y_v2
    epochs=50,                                 # ← more epochs
    batch_size=4096,
    class_weight=CLASS_WEIGHTS_CNN,            # ← 5x cap like CNN
    callbacks=[callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
)
evaluate_model(hybrid1, X_hyb1_test, y_te2, "Hybrid_XGB+CNN")
hybrid1.save("hybrid1_xgb_cnn.keras")

## NOTE !!!
**THIS ONE IS GOOD 98% AND FAAAST, NOT EVEN A FULL 2 MINUTES**

# Hybrid model 2 : XG x RF x MLP (Ensemble)

In [ ]:
# ── HYBRID 2: XGBoost + RF + MLP (Stacking) ──────────────────────────────────

# RF probabilities — retrain on 10-class filtered data
print("Retraining RF for stacking...")
rf_idx2 = np.random.choice(len(X_tr2), min(500_000, len(X_tr2)), replace=False)
rf_full = RandomForestClassifier(
    n_estimators=300,               # ← tuned: more trees
    max_depth=25,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced_subsample",  # ← tuned version
    n_jobs=-1,
    random_state=SEED
)
rf_full.fit(X_tr2[rf_idx2], y_tr2[rf_idx2])

rf_train_prob = rf_full.predict_proba(X_tr2)   # ← X_train → X_tr2
rf_val_prob   = rf_full.predict_proba(X_v2)    # ← X_val   → X_v2
rf_test_prob  = rf_full.predict_proba(X_te2)   # ← X_test  → X_te2

# MLP probabilities — wait for mlp_model to finish training first
print("Getting MLP probabilities...")
mlp_train_prob = mlp_model.predict(X_tr2, batch_size=8192)   # ← best_mlp → mlp_model
mlp_val_prob   = mlp_model.predict(X_v2,  batch_size=8192)
mlp_test_prob  = mlp_model.predict(X_te2, batch_size=8192)

# XGB probs — reuse xgb_full from Hybrid 1 (already trained on 10 classes)
# no need to retrain

# Stack: [XGB_probs | RF_probs | MLP_probs] → (30,) vector per sample
X_hyb2_train = np.concatenate([xgb_train_prob, rf_train_prob, mlp_train_prob], axis=1)
X_hyb2_val   = np.concatenate([xgb_val_prob,   rf_val_prob,   mlp_val_prob],   axis=1)
X_hyb2_test  = np.concatenate([xgb_test_prob,  rf_test_prob,  mlp_test_prob],  axis=1)

print(f"Stacked input shape: {X_hyb2_train.shape}")  # should be (N, 30) — 3×10 classes

# Meta-learner — deeper head since input is richer (3 models × 10 probs)
def build_meta_head2(input_dim, num_classes):
    inp = layers.Input(shape=(input_dim,))
    x = layers.Dense(256, activation="relu")(inp)  # ← wider: 128→256
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)    # ← extra layer
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model(inp, out, name="XGB_RF_MLP_Hybrid")

hybrid2 = build_meta_head2(X_hyb2_train.shape[1], NUM_CLASSES_2)  # ← NUM_CLASSES_2
hybrid2.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4),   # ← same lr
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
hybrid2.fit(
    X_hyb2_train, y_tr2,                        # ← y_train → y_tr2
    validation_data=(X_hyb2_val, y_v2),         # ← y_val → y_v2
    epochs=50,
    batch_size=4096,
    class_weight=CLASS_WEIGHTS_CNN,             # ← 5x cap
    callbacks=[callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
)
evaluate_model(hybrid2, X_hyb2_test, y_te2, "Hybrid_XGB+RF+MLP")
hybrid2.save("hybrid2_xgb_rf_mlp.keras")

## NOTE!!

kinda biased this one should take the crown but its kinda underperformed compared to it with the mini dataset before, still showed actaul W results 96%, witha  decent time of traning, i think sub 5 minutes, but still the same problem across aaaaaaall models, they cant identify intrusions, we kinda need another another apraoch, eh figures..

# comparison between the 2 hybrids

In [ ]:
# ── HYBRID MODELS COMPARISON ──────────────────────────────────────────────────
hybrid_results = {}
for name, model, Xte in [
    ("XGB + CNN",      hybrid1, X_hyb1_test),
    ("XGB + RF + MLP", hybrid2, X_hyb2_test),
]:
    preds = np.argmax(model.predict(Xte, batch_size=8192), axis=1)
    hybrid_results[name] = round((preds == y_te2).mean() * 100, 2)  # ← y_test → y_te2

print("\n" + "="*40)
print("   Hybrid Models Comparison")
print("="*40)
for name, acc in sorted(hybrid_results.items(), key=lambda x: -x[1]):
    print(f"  {name:<25}: {acc:.2f}%")
print("="*40)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(hybrid_results.keys(), hybrid_results.values(),   # ← fixed broken link
              color=["#9467BD", "#E377C2"])
ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Hybrid Models — CSE-CIC-IDS2018 (10 classes)")
ax.set_ylim(0, 105)
for bar, val in zip(bars, hybrid_results.values()):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.3, f"{val}%", ha="center")
plt.tight_layout()
plt.savefig("hybrid_comparison.png", dpi=100)
plt.show()                                                       # ← fixed broken link

# final comparison between all models:

In [ ]:
# ── FINAL COMPARISON — ALL MODELS ─────────────────────────────────────────────
all_results = {}

# Neural networks
for name, model, Xte in [
    ("MLP",  mlp_model,  X_te2),        # ← best_mlp → mlp_model, X_test → X_te2
    ("CNN",  cnn_model,  X_test_cnn),   # ← best_cnn → cnn_model
    ("LSTM", lstm_model, X_test_cnn),
]:
    preds = np.argmax(model.predict(Xte, batch_size=8192), axis=1)
    all_results[name] = round((preds == y_te2).mean() * 100, 2)  # ← y_test → y_te2

# Classical ML — tuned versions only for fair comparison
for name, model, Xte in [
    ("RF (tuned)",       rf2,   X_te2),
    ("XGBoost (tuned)",  xgb2,  X_te2),
    ("LightGBM (tuned)", lgb2,  X_te2),
]:
    preds = model.predict(Xte)
    all_results[name] = round((preds == y_te2).mean() * 100, 2)

# Hybrids
for name, model, Xte in [
    ("XGB + CNN",      hybrid1, X_hyb1_test),
    ("XGB + RF + MLP", hybrid2, X_hyb2_test),
]:
    preds = np.argmax(model.predict(Xte, batch_size=8192), axis=1)
    all_results[name] = round((preds == y_te2).mean() * 100, 2)  # ← y_test → y_te2

# Sort by accuracy descending
all_results = dict(sorted(all_results.items(), key=lambda x: -x[1]))

print("\n" + "="*45)
print("        Final Model Comparison")
print("="*45)
for name, acc in all_results.items():
    bar = "█" * int(acc / 2)
    print(f"  {name:<25}: {acc:.2f}%  {bar}")
print("="*45)

colors = {
    "MLP":             "#4C72B0",
    "CNN":             "#DD8452",
    "LSTM":            "#55A868",
    "RF (tuned)":      "#C44E52",    # ← updated key names
    "XGBoost (tuned)": "#8172B2",
    "LightGBM (tuned)":"#937860",
    "H1: XGB + CNN":       "#DA8BC3",
    "H2: XGB + RF + MLP":  "#8C8C8C",
}

fig, ax = plt.subplots(figsize=(13, 5))
bar_colors = [colors.get(n, "#333333") for n in all_results.keys()]
bars = ax.bar(all_results.keys(), all_results.values(), color=bar_colors)  # ← fixed broken link
ax.set_ylabel("Test Accuracy (%)")
ax.set_title("All Models — CSE-CIC-IDS2018 Final Comparison (10 classes)")  # ← updated title
ax.set_ylim(0, 108)
plt.xticks(rotation=30, ha="right")
for bar, val in zip(bars, all_results.values()):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.3, f"{val}%", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("final_comparison_all.png", dpi=100)
plt.show()  # ← fixed broken link

final note before the next step of the operation [what is teh next step of the operation ? aaah meme]

jk jk, so hybrids werent as efficient as they were in the old dataset, but now we know that the RF CNN and MLP are the go to for the next phase of the operation hehe, ok ok time to get serious..

some Key takeaways i noticed:
the GOats (97-98%) --> CNN, MLP, XGB+CNN hybrid are essentially tied. The difference between 97.98% and 97.89% is statistically negligible on 149,980 samples that's about 135 samples difference.

Middle offs (91-95%) —-> Classical ML performed surprisingly well considering they only trained on 300K samples vs the NNs' 700K. RF at 94.80% is genuinely impressive for a tree-based model.

and the Lgbm even tuned with LSTM sucked, thats it. ik so uuuh LSTM isnt performative in this senario even at large samples and also a bigger window of traning, and the GMB needed mooore rounds for it to converge but we can just take the other 3 that worked well.

alright guys heres what you should do EXACTLY, NO PHUSE NO MUSS:

- Classical ML (baseline + tuned) [done]
- NN models (CNN, MLP, LSTM) [done]
- Hybrid models  [done]
- Feature engineering
- Binary Infilteration detector
- Retrain everything with enriched features

good luck and have fun